<a href="https://colab.research.google.com/github/jcarlosfelix/jupyter_notebook/blob/main/Taller_RegEx_Limpieza_Datos_IAD2403.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Práctico: Limpieza y Manipulación de Datos con Expresiones Regulares (RegEx) en Python
**Materia:** Analítica de Datos (IAD-2403)  
**Institución:** Tecnológico Nacional de México / Instituto Tecnológico de Tijuana  
**Tema:** Preprocesamiento de Datos - Búsqueda, Validación y Corrección de Cadenas con RegEx

---

## Objetivos del Taller
1. **Comprender la sintaxis básica y avanzada** de las Expresiones Regulares (RegEx) mediante la librería estándar `re` de Python.
2. **Validar y estandarizar datos desestructurados o con inconsistencias de formato** comunes en proyectos reales de analítica (fechas, correos, RFC, CURP).
3. **Integrar RegEx con Pandas** para realizar operaciones masivas de limpieza sobre DataFrames de manera eficiente.

---
## 1. Fundamentos de RegEx en Python (`import re`)

Las Expresiones Regulares (RegEx) son secuencias de caracteres que forman un patrón de búsqueda. Nos permiten verificar si una cadena contiene un patrón determinado, extraer información específica o reemplazar partes del texto.

### Principales Funciones del Módulo `re`:
- `re.search(pattern, string)`: Busca la primera ocurrencia del patrón en la cadena.
- `re.match(pattern, string)`: Busca la coincidencia únicamente al inicio de la cadena.
- `re.findall(pattern, string)`: Devuelve una lista con todas las coincidencias del patrón.
- `re.sub(pattern, replacement, string)`: Reemplaza una o más coincidencias del patrón por un nuevo texto.

In [ ]:
import re
import pandas as pd
import numpy as np

# Ejemplo básico de búsqueda y extracción
texto_ejemplo = "El alumno con matrícula 20210458 asistió a la clase el día 15/09/2026."

# Buscar secuencia de dígitos (matrícula de 8 dígitos)
matricula = re.search(r'\b\d{8}\b', texto_ejemplo)
if matricula:
    print(f"Matrícula encontrada: {matricula.group()}")

### Tabla de Metacaracteres Más Frecuentes

| Sintaxis | Significado | Ejemplo de Coincidencia |
| :--- | :--- | :--- |
| `\d` | Cualquier dígito numérico `[0-9]` | `re.findall(r'\d+', 'Tengo 24 años')` -> `['24']` |
| `\w` | Cualquier carácter alfanumérico `[a-zA-Z0-9_]` | `re.findall(r'\w+', 'hola_123!')` -> `['hola_123']` |
| `\s` | Cualquier espacio en blanco (espacio, tab, salto de línea) | `\s+` |
| `+` | 1 o más repeticiones del carácter previo | `\d+` |
| `*` | 0 o más repeticiones | `a*` |
| `?` | 0 o 1 repetición (opcional) | `https?` (coincide con http y https) |
| `{n,m}` | Entre `n` y `m` repeticiones | `\d{2,4}` |
| `[ABC]` | Conjunto de caracteres permitidos | `[aeiou]` |
| `(^ / $)` | Inicio (`^`) y Fin (`$`) de la cadena | `^Tijuana$` |

---
## 2. Caso Práctico 1: Normalización de Fechas Inconsistentes

Uno de los errores más frecuentes en los datasets reales es la presencia de fechas capturadas en múltiples formatos (`YYYY-MM-DD`, `DD/MM/YYYY`, `DD-Mon-YY`, etc.).

In [ ]:
# Lista de fechas con formatos mixtos e inconsistentes
fechas_sucias = [
    "2026-09-15",
    "15/09/2026",
    "16-09-2026",
    "17/09/26",
    "09/18/2026", # Formato MM/DD/YYYY
    "2026/09/19"
]

def estandarizar_fecha(fecha_str):
    # Patrón 1: YYYY-MM-DD o YYYY/MM/DD -> ISO
    match_iso = re.match(r'^(\d{4})[-/](\d{2})[-/](\d{2})$', fecha_str)
    if match_iso:
        ano, mes, dia = match_iso.groups()
        return f"{ano}-{mes}-{dia}"

    # Patrón 2: DD/MM/YYYY o DD-MM-YYYY
    match_latam = re.match(r'^(\d{2})[-/](\d{2})[-/](\d{4})$', fecha_str)
    if match_latam:
        dia, mes, ano = match_latam.groups()
        return f"{ano}-{mes}-{dia}"

    # Patrón 3: DD/MM/YY (Año a 2 dígitos)
    match_corto = re.match(r'^(\d{2})[-/](\d{2})[-/](\d{2})$', fecha_str)
    if match_corto:
        dia, mes, ano = match_corto.groups()
        return f"20{ano}-{mes}-{dia}"

    return "Fecha_Invalida"

# Aplicar la función a la lista
fechas_limpias = [estandarizar_fecha(f) for f in fechas_sucias]
for original, limpia in zip(fechas_sucias, fechas_limpias):
    print(f"{original:<15}  --->   {limpia}")

---
## 3. Caso Práctico 2: Validación y Limpieza de Correos Electrónicos

La captura manual de correos suele introducir espacios innecesarios, caracteres especiales prohibidos o dominios duplicados (p. ej. `@gmail..com`).

In [ ]:
# Patrón general para un correo electrónico estándar
regex_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

correos_prueba = [
    "   juan.perez@tijuana.tecnm.mx  ", # Espacios al inicio/final
    "maria_gomez@gmail.com",          # Válido
    "carlos.invalid@domain",           # Dominio corto / sin TLD válido
    "sofia@@hotmail.com",              # Doble arroba
    "luis.rodriguez@gmail..com"         # Doble punto en dominio
]

def validar_y_limpiar_email(email):
    # 1. Eliminar espacios y pasar a minúsculas
    email_clean = email.strip().lower()

    # 2. Corregir errores comunes con re.sub()
    email_clean = re.sub(r'@{2,}', '@', email_clean)      # Corregir @@ -> @
    email_clean = re.sub(r'\.{2,}', '.', email_clean)    # Corregir .. -> .

    # 3. Validar con la expresión regular
    if re.match(regex_email, email_clean):
        return email_clean, "Válido"
    else:
        return email_clean, "Inválido"

for c in correos_prueba:
    limpio, estado = validar_y_limpiar_email(c)
    print(f"Original: {c:<35} | Limpio: {limpio:<30} | Estado: {estado}")

---
## 4. Caso Práctico 3: Validación de Estructuras Complejas (RFC y CURP)

En México, las claves alfanuméricas institucionales como el **RFC** (Registro Federal de Contribuyentes) y la **CURP** (Clave Única de Registro de Población) siguen una estructura oficial estricta que podemos verificar usando RegEx.

### Estructura del RFC Persona Física (13 Caracteres):
- **4 Letras:** Iniciales del nombre y apellidos (`[A-Z]{4}`).
- **6 Dígitos:** Fecha de nacimiento `YYMMDD` (`\d{6}`).
- **3 Caracteres:** Homoclave (`[A-Z0-9]{3}`).

### Estructura de la CURP (18 Caracteres):
- **4 Letras:** Nombre y apellidos (`[A-Z]{4}`).
- **6 Dígitos:** Fecha de nacimiento `YYMMDD` (`\d{6}`).
- **1 Letra:** Sexo (`[HM]`).
- **2 Letras:** Entidad federativa de nacimiento (`[A-Z]{2}`).
- **3 Letras:** Consonantes internas de apellidos y nombre (`[A-Z]{3}`).
- **2 Caracteres:** Homoclave/Diferenciador (`[A-Z0-9]{2}`).

In [ ]:
# Expresión regular oficial simplificada para RFC Persona Física
regex_rfc = r'^[A-Z&Ñ]{4}\d{6}[A-Z0-9]{3}$'

# Expresión regular oficial simplificada para CURP
regex_curp = r'^[A-Z]{4}\d{6}[HM][A-Z]{2}[B-DF-HJ-NP-TV-Z]{3}[A-Z0-9]{2}$'

ejemplos_rfc = ["GOME950412H89", "RODA80010112", "INVALID123", "MART010203XXX"]
ejemplos_curp = ["GOME950412HBCMRL09", "RODA800101MDFRXX01", "123456789012345678"]

print("--- VALIDACIÓN DE RFC ---")
for rfc in ejemplos_rfc:
    es_valido = bool(re.match(regex_rfc, rfc.upper()))
    print(f"RFC: {rfc:<15} -> {'✓ VÁLIDO' if es_valido else '✗ INVÁLIDO'}")

print("\n--- VALIDACIÓN DE CURP ---")
for curp in ejemplos_curp:
    es_valido = bool(re.match(regex_curp, curp.upper()))
    print(f"CURP: {curp:<20} -> {'✓ VÁLIDA' if es_valido else '✗ INVÁLIDA'}")

---
## 5. Integración de RegEx con Pandas en DataFrames

Pandas cuenta con el accesor `.str` que permite aplicar patrones RegEx directamente a columnas completas mediante métodos como:
- `df['col'].str.contains(pattern)`: Filtra filas que coinciden con el patrón.
- `df['col'].str.extract(pattern)`: Extrae grupos definidos por paréntesis `()`.
- `df['col'].str.replace(pattern, replacement, regex=True)`: Reemplaza patrones en toda la columna.

In [ ]:
# Dataset de prueba desestructurado
datos = {
    'Alumno': ['Osnay', 'Guille', 'Melanny', 'Win', 'Jayt'],
    'Telefono': ['(664) 123-4567', '664.987.6543', '+52 664 111 2233', '6641239988', '123-456'],
    'Correo': ['osnay@tecnm.mx', 'guille@gmail..com', 'melanny@hotmail.com', 'win_123@yahoo.com', 'jayt@site'],
    'Fecha_Registro': ['15/09/2026', '2026-09-16', '17-Sep-26', '18/09/2026', 'Invalid_Date']
}

df = pd.DataFrame(datos)
print("DataFrame Original:")
print(df)

# 1. Extraer solo los 10 dígitos numéricos del teléfono con str.replace
df['Telefono_Limpio'] = df['Telefono'].str.replace(r'\D', '', regex=True)

# 2. Corregir dominios de correo con doble punto
df['Correo_Limpio'] = df['Correo'].str.replace(r'\.{2,}', '.', regex=True)

# 3. Filtrar correos que cumplen con el formato estándar
df['Correo_Valido'] = df['Correo_Limpio'].str.contains(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', regex=True)

print("\nDataFrame Limpio:")
print(df[['Alumno', 'Telefono_Limpio', 'Correo_Limpio', 'Correo_Valido']])

---
## 6. Actividades / Retos Prácticos para los Alumnos

A continuación, resuelve los siguientes 3 ejercicios aplicando lo aprendido durante el taller.

### Reto 1: Extracción de Claves de Asignaturas
Dada una lista de códigos de materias, utiliza `re.findall()` o `re.extract()` para extraer únicamente la clave oficial de la materia con formato de 3 letras mayúsculas, un guion y 4 números (Ejemplo: `IAD-2403`, `SCD-1015`).

In [ ]:
textos_materias = [
    "Inscrito en la materia Analítica de Datos (IAD-2403) grupo A",
    "Sistemas Operativos clave SCD-1015 impartida en aula 3",
    "Curso de verano Matemáticas I - no tiene clave estándar",
    "Taller de Base de Datos (AEB-1055) - Aula Virtual"
]

# ESCRIBE TU CÓDIGO AQUÍ:
patron_clave = r'[A-Z]{3}-\d{4}'

for texto in textos_materias:
    coincidencias = re.findall(patron_clave, texto)
    print(f"Texto: {texto:<60} -> Clave: {coincidencias}")

### Reto 2: Normalización de Nombres con Tildes o Caracteres Especiales
A veces los usuarios escriben nombres con números o caracteres extraños por error (ej. `M4ria`, `Jua&n`, `S0fia`). Crea un patrón RegEx para eliminar cualquier carácter que NO sea una letra (incluyendo vocales con acento y eñes) o un espacio.

In [ ]:
nombres_sucios = ["M4ria123", "Jua&n P3rez", "Sofía López!!", "C4rlos_88"]

# ESCRIBE TU CÓDIGO AQUÍ:
for nombre in nombres_sucios:
    # Patrón que reemplaza todo lo que no sea letra o espacio por cadena vacía
    limpio = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑ\s]', '', nombre)
    print(f"Original: {nombre:<20} -> Limpio: {limpio}")

### Reto 3: Validación Masiva en Pandas
Carga el dataset de estudiantes generado en clase (`IAD-2403_Dataset_Ruleta_Errores_Estudiantes_1000.csv`) y utiliza Pandas + RegEx para identificar cuántas filas tienen un formato de `Fecha de Captura` inválido o no estandarizado.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ:
try:
    df_estudiantes = pd.read_csv('IAD-2403_Dataset_Ruleta_Errores_Estudiantes_1000.csv')

    # Patrón para formato estricto ISO YYYY-MM-DD o DD/MM/YYYY
    patron_fecha_valida = r'^(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4})$'

    fechas_invalidas = df_estudiantes[
        ~df_estudiantes['Fecha de Captura'].fillna('').str.match(patron_fecha_valida)
    ]

    print(f"Total de filas con formato de fecha atípico/inválido: {len(fechas_invalidas)}")
    print("\nEjemplo de fechas encontradas:")
    print(fechas_invalidas[['No.', 'ID / Nombre u Alias', 'Fecha de Captura']].head(10))
except FileNotFoundError:
    print("El archivo 'IAD-2403_Dataset_Ruleta_Errores_Estudiantes_1000.csv' no fue encontrado en el directorio actual.")